# Học phân loại văn bản với dữ liệu tin nhắn SMS spam

Notebook này dạy từng bước: biến văn bản thành số (bag of words, TF-IDF), Naive Bayes, logistic regression cho văn bản, tìm tham số cho cả pipeline xử lý văn bản, đọc từ quan trọng và phân tích lỗi.

Dữ liệu: 5.572 tin nhắn SMS tiếng Anh, mỗi tin có nhãn ham (tin bình thường) hoặc spam (tin rác, quảng cáo, lừa đảo). Nguồn: SMS Spam Collection, UCI (Almeida và Hidalgo, 2011), bản sao trên GitHub của justmarkham.

Chuẩn bị:
1. Đặt file data10.csv cùng thư mục với notebook.
2. Cài thư viện nếu chưa có: pip install pandas numpy matplotlib scikit-learn

Cách làm: đọc phần giải thích, viết code vào ô trống, chạy bằng Shift + Enter, rồi so sánh với kết quả mong đợi. Nếu kết quả khác, đọc lại đề bài và thử lại trước khi xem đáp án.

## Bước 0: Import thư viện

Mục tiêu: nạp các thư viện dùng trong toàn bộ notebook. Ô này đã viết sẵn, bạn chỉ cần chạy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
pd.set_option('display.max_colwidth', 120)

## Bước 1: Đọc dữ liệu

Mục tiêu: đưa file CSV vào DataFrame và xem vài tin nhắn.

Kiến thức: pd.read_csv đọc file thành bảng. Cột message là văn bản tự do.

Kết quả mong đợi: (5572, 2).

In [ ]:
# Viết code ở đây:
# 1. Đọc file data10.csv vào biến df
# 2. In df.shape và 5 dòng đầu


## Bước 2: Tỷ lệ spam và tin trùng lặp

Mục tiêu: đếm số tin mỗi loại và số tin bị lặp lại.

Kiến thức: duplicated() đánh dấu các dòng giống hệt một dòng đã xuất hiện trước đó. Tin trùng lặp có thể rơi vào cả tập train và tập test, làm kết quả đánh giá đẹp hơn thực tế.

Kết quả mong đợi: ham có 4825 tin, spam có 747 tin, tỷ lệ spam khoảng 0,134. Có 403 dòng trùng lặp, trong đó 309 ham và 94 spam.

Câu hỏi: có nên xóa tin trùng lặp không? Trong thực tế, tin spam hàng loạt có hay bị gửi lặp lại không?

In [ ]:
# Viết code ở đây:
# 1. In value_counts của label và tỷ lệ spam (làm tròn 3 chữ số)
# 2. In số dòng trùng lặp (duplicated().sum())
# 3. In số tin trùng lặp theo từng label


## Bước 3: Độ dài tin nhắn

Mục tiêu: tạo feature đơn giản đầu tiên từ văn bản.

Kiến thức: str.len() đếm số ký tự. Một feature đơn giản đôi khi đã phân biệt được hai lớp khá tốt.

Kết quả mong đợi: tin spam dài hơn: trung vị 149 ký tự so với 52 ký tự của ham. Spam tập trung từ 133 đến 157 ký tự, gần giới hạn 160 ký tự của một SMS.

In [ ]:
# Viết code ở đây:
# 1. df['length'] = df['message'].str.len()
# 2. In describe() của length theo label (làm tròn 1 chữ số)
# 3. Vẽ hai histogram chồng lên nhau (bins=50, alpha=0.6, giới hạn trục x từ 0 đến 300)


## Bước 4: Chia train/test

Mục tiêu: giữ 20% làm tập test, giữ nguyên tỷ lệ spam.

Kiến thức: y = 1 nếu là spam. X ở bài này là một Series văn bản, không phải bảng số. Bước biến văn bản thành số sẽ được học chỉ từ tập train.

Kết quả mong đợi: X_train có 4457 tin, X_test có 1115 tin (149 spam).

In [ ]:
# Viết code ở đây:
# 1. X = df['message'], y = (df['label'] == 'spam').astype(int)
# 2. train_test_split với test_size=0.2, random_state=42, stratify=y
# 3. In số tin mỗi tập và số spam trong y_test


## Bước 5: Bag of words

Mục tiêu: biến mỗi tin nhắn thành một vector đếm số lần xuất hiện của từng từ.

Kiến thức: CountVectorizer làm 3 việc: tách văn bản thành từ (tokenize), chuyển về chữ thường, và đếm. Kết quả là một ma trận thưa (sparse matrix): mỗi dòng là một tin, mỗi cột là một từ trong từ điển, phần lớn ô bằng 0. Chỉ fit trên tập train. Từ chưa gặp trong tập test sẽ bị bỏ qua.

Kết quả mong đợi: ma trận train có kích thước (4457, 7668), tức là từ điển có 7668 từ. Chỉ khoảng 0,17% số ô khác 0. Tin đầu tiên 'He will, you guys close?' cho ra 5 từ: he, will, you, guys, close.

Câu hỏi: tại sao lưu dưới dạng ma trận thưa thay vì DataFrame bình thường?

In [ ]:
# Viết code ở đây:
# 1. cv = CountVectorizer(), X_train_cv = cv.fit_transform(X_train), X_test_cv = cv.transform(X_test)
# 2. In kích thước X_train_cv và số từ trong từ điển (len(cv.vocabulary_))
# 3. In tỷ lệ ô khác 0: X_train_cv.nnz / (số dòng x số cột)
# 4. Với tin đầu tiên của X_train, in các từ và số lần xuất hiện (gợi ý: cv.inverse_transform)


## Bước 6: Multinomial Naive Bayes

Mục tiêu: huấn luyện mô hình kinh điển cho phân loại văn bản.

Kiến thức: MultinomialNB tính xác suất mỗi từ xuất hiện trong spam và trong ham, rồi dùng định lý Bayes để kết hợp. Nó giả định các từ độc lập với nhau. Giả định này sai, nhưng mô hình vẫn chạy nhanh và thường rất tốt với văn bản. Hàm evaluate dưới đây in các chỉ số cần thiết.

Kết quả mong đợi: accuracy khoảng 0,987, precision khoảng 0,986, recall khoảng 0,919, F1 khoảng 0,951. Có 2 FP và 12 FN.

Câu hỏi: với bộ lọc spam, FP nghĩa là một tin nhắn thật bị đưa vào thư mục spam. FP hay FN gây khó chịu hơn cho người dùng?

In [ ]:
def evaluate(name, y_true, pred):
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    res = {'Mô hình': name, 'accuracy': accuracy_score(y_true, pred), 'precision': precision_score(y_true, pred),
           'recall': recall_score(y_true, pred), 'F1': f1_score(y_true, pred), 'FP': fp, 'FN': fn}
    print(' | '.join(f'{k}: {v:.3f}' if isinstance(v, float) else f'{k}: {v}' for k, v in res.items()))
    return res

# Viết code ở đây:
# 1. nb = MultinomialNB(), fit trên X_train_cv, y_train
# 2. pred_nb = nb.predict(X_test_cv)
# 3. Gọi evaluate('NB + count', y_test, pred_nb), lưu vào res_nb


## Bước 7: TF-IDF

Mục tiêu: cho từ hiếm và đặc trưng trọng số cao hơn từ phổ biến.

Kiến thức: TF (term frequency) là số lần từ xuất hiện trong tin. IDF (inverse document frequency) lớn khi từ xuất hiện trong ít tin. TF-IDF = TF x IDF, nên từ như "the", "you" có trọng số thấp, còn từ như "prize", "claim" có trọng số cao. TfidfVectorizer còn chuẩn hóa mỗi vector về độ dài 1, nên tin dài và tin ngắn so sánh được với nhau.

Kết quả mong đợi: tin spam đầu tiên là quảng cáo FREE NOKIA. Từ có TF-IDF cao nhất là free (khoảng 0,362), sau đó là các từ hiếm như upto, d3wv, 2price. Các từ có IDF thấp nhất là to, you, the, in, and.

In [ ]:
# Viết code ở đây:
# 1. tfidf = TfidfVectorizer(), X_train_tf = fit_transform(X_train), X_test_tf = transform(X_test)
# 2. Với một tin spam trong X_train (ví dụ dòng đầu tiên có y_train bằng 1), in 5 từ có TF-IDF cao nhất
# 3. In 5 từ có IDF thấp nhất (phổ biến nhất) dùng tfidf.idf_ và tfidf.get_feature_names_out()


## Bước 8: Logistic regression với TF-IDF

Mục tiêu: so sánh logistic regression với Naive Bayes.

Kiến thức: Pipeline gộp TfidfVectorizer và mô hình. Khi đó bạn có thể đưa thẳng văn bản thô vào fit và predict, và cross-validation sẽ fit lại TF-IDF trong từng fold. Logistic regression có tham số C mặc định bằng 1, khá mạnh với văn bản có hàng nghìn cột.

Kết quả mong đợi: accuracy khoảng 0,973, precision bằng 1,0 (không có FP), nhưng recall chỉ khoảng 0,799 (bỏ sót 30 tin spam), F1 khoảng 0,888. Kém NB ở bước 6.

In [ ]:
# Viết code ở đây:
# 1. lr = Pipeline([('tfidf', TfidfVectorizer()), ('model', LogisticRegression(max_iter=1000))])
# 2. Fit trên X_train (văn bản thô), dự đoán X_test
# 3. Gọi evaluate('LR + tfidf', ...), lưu res_lr


## Bước 9: Tìm tham số cho cả pipeline

Mục tiêu: dùng GridSearchCV để chọn cùng lúc cách tách từ và độ mạnh regularization.

Kiến thức: ngram_range=(1, 2) thêm cặp hai từ liền nhau (bigram) như "free entry", "call now". min_df=2 bỏ các từ chỉ xuất hiện trong 1 tin. Tham số của bước trong pipeline có dạng tên_bước__tên_tham_số. Bước này mất khoảng 30 giây.

Kết quả mong đợi: best params là C = 100, min_df = 1 và ngram_range = (1, 2), CV F1 khoảng 0,95. Trên tập test: recall khoảng 0,906, F1 khoảng 0,944, 2 FP và 14 FN.

In [ ]:
# Viết code ở đây:
# 1. param_grid = {'tfidf__ngram_range': [(1, 1), (1, 2)], 'tfidf__min_df': [1, 2], 'model__C': [1, 10, 100]}
# 2. cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# 3. GridSearchCV(lr, param_grid, cv=cv, scoring='f1', n_jobs=-1), fit trên X_train
# 4. In best_params_ và best_score_, rồi gọi evaluate cho best_estimator_ trên tập test, lưu res_best


## Bước 10: Từ nào báo hiệu spam

Mục tiêu: đọc hệ số của logistic regression để giải thích mô hình.

Kiến thức: mỗi từ (hoặc cặp từ) có một hệ số. Hệ số dương lớn đẩy tin nhắn về phía spam, hệ số âm đẩy về phía ham.

Kết quả mong đợi: các từ spam mạnh nhất là txt (khoảng 14,4), text, uk, free, call, stop, reply, www, claim. Các từ ham mạnh nhất là my, ok, me, gt, lt, home, sir, da.

Câu hỏi: những từ này có hợp lý không? Kẻ gửi spam có thể né bộ lọc bằng cách nào?

In [ ]:
# Viết code ở đây:
# 1. Lấy tên feature từ best.named_steps['tfidf'].get_feature_names_out()
# 2. Lấy hệ số từ best.named_steps['model'].coef_[0]
# 3. Tạo Series và in 15 từ có hệ số lớn nhất và 15 từ có hệ số nhỏ nhất (làm tròn 2 chữ số)


## Bước 11: Phân tích lỗi

Mục tiêu: đọc các tin nhắn bị phân loại sai để hiểu giới hạn của mô hình.

Kiến thức: lọc X_test theo điều kiện y thật khác y dự đoán. FP là tin ham bị đoán là spam. FN là tin spam bị bỏ sót.

Kết quả mong đợi: có 2 FP, ví dụ 'Alright i have a new goal now'. Có 14 FN, nhiều tin không có từ khóa spam điển hình, ví dụ 'Latest News! Police station toilet stolen...' hoặc tin quảng cáo bán phi tiêu.

In [ ]:
# Viết code ở đây:
# 1. Tạo errors = pd.DataFrame({'message': X_test, 'thật': y_test, 'đoán': pred_best})
# 2. In các dòng có thật = 0, đoán = 1 (FP)
# 3. In các dòng có thật = 1, đoán = 0 (FN)


## Bước 12: Dự đoán tin nhắn mới

Mục tiêu: dùng mô hình cho tin nhắn bạn tự viết.

Kiến thức: vì pipeline chứa cả bước TF-IDF, chỉ cần đưa list văn bản vào predict_proba.

Kết quả mong đợi: hai tin giống spam có xác suất cao (khoảng 1,000 và 0,943). Hai tin bình thường có xác suất rất thấp (khoảng 0,001 và 0,004).

In [ ]:
# Viết code ở đây:
# 1. Tạo list new_msgs gồm ít nhất 4 tin nhắn (2 tin giống spam, 2 tin bình thường)
# 2. In xác suất spam của mỗi tin (best.predict_proba(new_msgs)[:, 1], làm tròn 3 chữ số)


## Bước 13: Bảng so sánh các mô hình

Mục tiêu: ghép kết quả các mô hình vào một bảng.

Kiến thức: thêm một mô hình nữa để so sánh: Naive Bayes với TF-IDF và LinearSVC (SVM tuyến tính, thường rất mạnh với văn bản).

Kết quả mong đợi: NB + count có F1 cao nhất (khoảng 0,951) với 2 FP và 12 FN. LR tuned (F1 khoảng 0,944) và LinearSVC (F1 khoảng 0,941) gần bằng. NB + tfidf có recall thấp nhất (khoảng 0,705) nhưng không có FP nào.

Câu hỏi: mô hình nào bạn sẽ dùng cho bộ lọc spam trên điện thoại? Hãy tính đến cả số FP, không chỉ F1.

In [ ]:
# Viết code ở đây:
# 1. Fit Pipeline(TfidfVectorizer(), MultinomialNB()) trên X_train, evaluate, lưu res_nb_tf
# 2. Fit Pipeline(TfidfVectorizer(ngram_range=(1, 2)), LinearSVC()) trên X_train, evaluate, lưu res_svm
# 3. Tạo DataFrame từ res_nb, res_nb_tf, res_lr, res_best, res_svm và in ra (làm tròn 3 chữ số)


## Bước 14: Tự thử nghiệm

Không có code mẫu ở bước này. Hãy tự chọn ít nhất hai việc sau:

1. Xóa tin trùng lặp (df.drop_duplicates()) trước khi chia train/test, rồi chạy lại mô hình tốt nhất. F1 trên tập test thay đổi thế nào?
2. Thêm stop_words='english' vào TfidfVectorizer. Kết quả tốt hơn hay kém hơn? Vì sao các từ như "you", "your" có thể hữu ích cho việc nhận ra spam?
3. Thử TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5)), tức là dùng cụm ký tự thay vì từ. Cách này có bắt được spam viết sai chính tả như "fr33" không?
4. Viết ba câu tiếng Anh tóm tắt kết quả cho phần "Results" của một báo cáo. Ví dụ: "The tuned logistic regression with word and bigram TF-IDF features reached an F1 score of ... on the test set, with ... false positives."
